<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "694"
iso_country = "Sierra Leone"
year = 2017
import_portfolio = "h3_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h3_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 5.820*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Sierra Leone': 

Number of trade records present in dataset: 13362
Number of partner countries present in dataset: 114
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1039.9m
Total weight within the dataset: 1751.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,694,Sierra Leone,364,Iran,Import,Sea,711510,711510 | Metal; catalysts in,7115,7115 | Articles of precious,71,"71 | Natural, cultured pe",3824,Jewellery and other articles of precious metal...,Mining and Quarrying,36,NFM - Metals n.e.c.,1765.887,NaN,1765.887,kg,1188.87,1188.87,1.765887,13512.0,8395.964952,14.826325,['Container'],['Container'],0.055816,0.055692,0.055705,0.055705,0.055761,0.055761,0.058560,0.106042,0.063089,0.060189,0.060381,0.061591,0.057262,0.102898,0.067298,0.062225,0.060631,0.063620,0.056123,0.114681,0.084174,0.073808,0.068019,0.066422,0.055006,0.121089,0.099311,0.091280,0.072607,0.071336,0.053909,0.124692,0.121060,0.123324,0.077179,0.074997,0.052791,0.121995,0.113924,0.123255,0.096833,0.076432,0.051737,0.106647,0.112884,0.117640,0.083559,0.077341,0.050663,0.098121,0.099071,0.112728,0.095435,0.077921,3824
1,1,694,Sierra Leone,364,Iran,Import,Sea,711590,711590 | Metal; precious or m,7115,7115 | Articles of precious,71,"71 | Natural, cultured pe",3824,Jewellery and other articles of precious metal...,Mining and Quarrying,42,OMF - Manufactures n.e.c.,1730.070,NaN,1730.070,kg,1188.88,1188.88,1.730070,13512.0,8395.964952,14.525607,['Container'],['Container'],0.054684,0.054562,0.054575,0.054575,0.054630,0.054630,0.057373,0.103891,0.061810,0.058968,0.059156,0.060342,0.056101,0.100811,0.065933,0.060963,0.059401,0.062329,0.054985,0.112355,0.082466,0.072311,0.066639,0.065075,0.053890,0.118633,0.097297,0.089429,0.071134,0.069889,0.052816,0.122163,0.118604,0.120823,0.075614,0.073476,0.051720,0.119521,0.111613,0.120755,0.094869,0.074882,0.050688,0.104484,0.110594,0.115254,0.081864,0.075772,0.049635,0.096130,0.097061,0.110442,0.093499,0.076341,3824


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $347.6m (33.4%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               194628303.0  341324000.0     18.7     19.5
22 | Beverages, spirits a   23466087.0   47684930.1      2.3      2.7
11 | Products of the mill   20747159.2   49534481.8      2.0      2.8
15 | Animal, vegetable or   17712156.7   19130952.5      1.7      1.1
17 | Sugars and sugar con   17213629.1   35165764.1      1.7      2.0
04 | Dairy produce; birds   14953966.4    6442358.2      1.4      0.4
21 | Miscellaneous edible   12942668.8    5171382.0      1.2      0.3
02 | Meat and edible meat   11154206.4    5968158.5      1.1      0.3
07 | Vegetables and certa    8906264.5   14750689.0      0.9      0.8
19 | Preparations of cere    7999358.4    3005132.9      0.8      0.2
20 | Preparations of vege    5573669.3    4445560.2      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $361.1m (34.7%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       271209171.5  438186137.0     26.1     25.0
Food & Beverages   85294847.2   97627609.7      8.2      5.6
Fishing             4609748.2    2220394.7      0.4      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $321.9m (31.0%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
PCR - Processed rice                          192416105.7  334226775.6   
OFD - Food products n.e.c.                     57492334.1   65176941.4   
VOL - Vegetable oils and fats                  17711311.4   19129059.1   
SGR - Sugar                                    15615865.3   34260463.6   
OMT - Meat products n.e.c.                     10804577.3    5623832.8   
MIL - Dairy products                           10013982.2    3639877.0   
V_F - Vegetables, fruit, nuts                   9389242.8   15204266.8   
OAP - Animal products n.e.c.                    5096471.7    2845441.6   
OCR - Crops n.e.c.                              1523609.9     240406.4   
CMT - Bovine meat prods                         1128584.9     515627.6   
PDR - Paddy rice                              

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 33.4% 
GTAP 31.0% 
EORA 34.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46419/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,694,Sierra Leone,364,Iran,Import,Sea,711510,711510 | Metal; catalysts in,7115,7115 | Articles of precious,71,"71 | Natural, cultured pe",3824,Jewellery and other articles of precious metal...,Mining and Quarrying,36,NFM - Metals n.e.c.,1765.887,NaN,1765.887,kg,1188.87,1188.87,1.765887,13512.0,8395.964952,14.826325,['Container'],['Container'],0.055816,0.055692,0.055705,0.055705,0.055761,0.055761,0.058560,0.106042,0.063089,0.060189,0.060381,0.061591,0.057262,0.102898,0.067298,0.062225,0.060631,0.063620,0.056123,0.114681,0.084174,0.073808,0.068019,0.066422,0.055006,0.121089,0.099311,0.091280,0.072607,0.071336,0.053909,0.124692,0.121060,0.123324,0.077179,0.074997,0.052791,0.121995,0.113924,0.123255,0.096833,0.076432,0.051737,0.106647,0.112884,0.117640,0.083559,0.077341,0.050663,0.098121,0.099071,0.112728,0.095435,0.077921,3824,NaN
1,1,694,Sierra Leone,364,Iran,Import,Sea,711590,711590 | Metal; precious or m,7115,7115 | Articles of precious,71,"71 | Natural, cultured pe",3824,Jewellery and other articles of precious metal...,Mining and Quarrying,42,OMF - Manufactures n.e.c.,1730.070,NaN,1730.070,kg,1188.88,1188.88,1.730070,13512.0,8395.964952,14.525607,['Container'],['Container'],0.054684,0.054562,0.054575,0.054575,0.054630,0.054630,0.057373,0.103891,0.061810,0.058968,0.059156,0.060342,0.056101,0.100811,0.065933,0.060963,0.059401,0.062329,0.054985,0.112355,0.082466,0.072311,0.066639,0.065075,0.053890,0.118633,0.097297,0.089429,0.071134,0.069889,0.052816,0.122163,0.118604,0.120823,0.075614,0.073476,0.051720,0.119521,0.111613,0.120755,0.094869,0.074882,0.050688,0.104484,0.110594,0.115254,0.081864,0.075772,0.049635,0.096130,0.097061,0.110442,0.093499,0.076341,3824,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $280.6m of $1039.9m (27.0%)
Volume: 458.4 of 1751.3 million kilograms (26.2%)
Value-to-Weight: 0.61 from 0.59 US$ per kg (+3.1%)
Average distance to market (volume-weighted): 6630.1 from 5579.5 nm (+18.8%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    516  225738251.4  388849694.6     21.7     22.2
1      Sugar and Honey     84   17213629.1   35165764.1      1.7      2.0
2           Vegetables    178   12374938.2   16726104.9      1.2      1.0
3        Meat and Prep     46    9964979.9    5439362.6      1.0      0.3
4                Dairy    125    9880999.3    3603009.2      1.0      0.2
5        Fats and Oils     97    3065482.6    1879183.8      0.3      0.1
6  Unprocessed Cereals     11    1689173.4    6188731.9      0.2      0.4
7               Fruits     53     652902.6     563070.0      0.1      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,9156949.6,1.656226e+07,9863526.5,9408923.7,9438909.9,9629467.0,8431580.6,1.946911e+07,1.890050e+07,1.925734e+07,12064190.5,11720131.6,7925266.5,1.532811e+07,1.547843e+07,1.761396e+07,1.490984e+07,12176024.5
1,Fats and Oils,4529091.8,8.180465e+06,4877697.8,4652106.4,4666908.8,4761992.6,4171431.5,9.611415e+06,9.330522e+06,9.508408e+06,5964300.7,5792790.3,3921781.3,7.571933e+06,7.647448e+06,8.704074e+06,7.366450e+06,6017504.2
2,Fruits,798541.0,1.438207e+06,859466.1,819789.5,822364.5,839304.9,735899.6,1.690954e+06,1.639822e+06,1.672200e+06,1051213.3,1019775.1,692176.0,1.333681e+06,1.347075e+06,1.532321e+06,1.296571e+06,1059079.2
3,Meat and Prep,9779219.2,1.770316e+07,10535002.9,10050502.3,10082566.2,10284946.3,9003046.7,2.081669e+07,2.020901e+07,2.058821e+07,12887714.2,12522112.0,8461271.6,1.638252e+07,1.654148e+07,1.882167e+07,1.593398e+07,13010037.6
4,Processed Cereals,711218292.5,1.114535e+09,746255074.0,708313463.6,710686423.4,735450612.5,671579365.5,1.333681e+09,1.276956e+09,1.321348e+09,938199673.9,845733969.5,643910455.1,1.099220e+09,1.121272e+09,1.263748e+09,1.073290e+09,866958353.8
5,Sugar and Honey,37149923.0,6.243342e+07,39467032.6,37550793.9,37673077.6,38721560.2,34669963.9,7.404274e+07,7.136766e+07,7.328883e+07,48963349.7,45760659.8,32938867.3,5.965777e+07,6.055417e+07,6.856911e+07,5.813524e+07,47217991.1
6,Unprocessed Cereals,10728499.3,1.757677e+07,11357467.3,10796646.7,10822525.5,11149070.3,10061630.2,2.064876e+07,1.963723e+07,2.040737e+07,13870568.3,13045164.1,9596315.6,1.689919e+07,1.718764e+07,1.948688e+07,1.638571e+07,13441169.7
7,Vegetables,25386558.3,4.532502e+07,27281436.2,26011043.9,26091088.8,26652950.8,23435615.0,5.328737e+07,5.158305e+07,5.268869e+07,33359724.3,32270470.5,22072500.3,4.215295e+07,4.260833e+07,4.846772e+07,4.098248e+07,33489563.1


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.75,0.07,0.03,0.03,0.05,1.12,1.06,1.10,0.37,0.33,0.75,0.76,0.98,0.71,0.43
1,Fats and Oils,1.19,0.11,0.04,0.04,0.08,1.77,1.68,1.74,0.58,0.53,1.19,1.22,1.56,1.12,0.68
2,Fruits,0.98,0.09,0.03,0.04,0.06,1.46,1.38,1.43,0.48,0.43,0.98,1.00,1.29,0.93,0.56
3,Meat and Prep,0.80,0.08,0.03,0.03,0.05,1.19,1.12,1.16,0.39,0.35,0.79,0.81,1.04,0.75,0.46
4,Processed Cereals,1.79,0.16,-0.01,-0.00,0.11,2.93,2.68,2.88,1.18,0.77,2.02,2.11,2.75,1.90,0.99
5,Sugar and Honey,1.47,0.13,0.02,0.03,0.09,2.29,2.13,2.24,0.83,0.64,1.55,1.60,2.07,1.46,0.83
6,Unprocessed Cereals,4.05,0.37,0.04,0.06,0.25,6.27,5.67,6.12,2.25,1.77,4.32,4.49,5.86,4.02,2.28
7,Vegetables,1.61,0.15,0.05,0.06,0.10,2.41,2.27,2.36,0.80,0.71,1.62,1.66,2.13,1.53,0.92
